In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().parents[0]  # notebooks/ -> repo root
sys.path.insert(0, str(REPO_ROOT / "src"))

import recipe_wrangler


In [2]:
from pathlib import Path
import recipe_wrangler

REPO_ROOT = Path(recipe_wrangler.__file__).resolve().parents[2]  # package -> src -> repo
REPO_ROOT

PosixPath('/home/karvanitis/RecipeWrangler-Backend')

## Check Chromadb Databases

In [3]:
from recipe_wrangler.utils.chroma_client import get_chroma_client, CHROMA_HOST, CHROMA_PORT

client = get_chroma_client()  # defaults: host=localhost, port=8000
collections = client.list_collections()

print(f"Collections at {CHROMA_HOST}:{CHROMA_PORT}")
for col in collections:
    print(f"- {col.name} (count={col.count()})")


2026-01-29 14:49:30.939156851 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Collections at localhost:8000
- common_units (count=7)
- ingredients (count=4076)
- foods_density_v1 (count=665)
- sustainability_ingredients (count=7373)
- nutritional_ingredients_irish (count=1307)


## Check Utilities

In [4]:
from recipe_wrangler.utils.query_chromadb import get_ingredient_embedding

get_ingredient_embedding("low-fat gouda")

/home/karvanitis/RecipeWrangler-Backend/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


[0.0177001953125,
 -0.005126953125,
 -0.012939453125,
 -0.0096435546875,
 0.0277099609375,
 -0.004364013671875,
 -0.01458740234375,
 -0.0201416015625,
 -0.0166015625,
 0.01275634765625,
 -0.00518798828125,
 -0.0016937255859375,
 0.039306640625,
 -0.0013275146484375,
 0.01019287109375,
 -0.0189208984375,
 0.0089111328125,
 0.0103759765625,
 0.0537109375,
 0.02978515625,
 0.031494140625,
 -0.025390625,
 0.0283203125,
 0.00390625,
 0.019775390625,
 0.01904296875,
 -0.0140380859375,
 -0.01446533203125,
 -0.0198974609375,
 0.0390625,
 -0.0252685546875,
 -0.0177001953125,
 0.0216064453125,
 0.0179443359375,
 -0.0289306640625,
 0.00860595703125,
 -0.0198974609375,
 0.01312255859375,
 0.0084228515625,
 -0.03125,
 -0.017578125,
 -0.0098876953125,
 -0.0225830078125,
 0.00909423828125,
 0.0150146484375,
 -0.0147705078125,
 -0.0203857421875,
 -0.010498046875,
 0.0177001953125,
 -0.00762939453125,
 -0.00616455078125,
 -0.0240478515625,
 -0.0023956298828125,
 0.0031280517578125,
 -0.0009765625,
 -0.

In [ ]:
from recipe_wrangler.utils.query_chromadb import query_sustainability_db

result = query_sustainability_db("low-fat gouda")
print(result)

In [ ]:
from recipe_wrangler.utils.query_chromadb import query_nutritional_db_irish

result = query_nutritional_db_irish("tomatoe")

print(result)

## Parse Recipe Tool

In [5]:
from recipe_wrangler.tools.parse_recipe_tool import parse_recipe_tool
import json

raw_text = """
Garlic Butter Shrimp

Ingredients (2 servings):

200g shrimp (peeled & deveined)

2 tbsp butter

3 garlic cloves, minced

Juice of half a lemon

Salt & pepper to taste

Fresh parsley (optional)

Instructions:

Heat butter in a skillet over medium heat.

Add garlic, cook 30 seconds until fragrant.

Toss in shrimp, season with salt & pepper.

Cook 2–3 minutes per side until pink.

Squeeze lemon juice on top and sprinkle parsley.

Serve with rice, pasta, or crusty bread.
"""

recipe_dict = parse_recipe_tool.invoke({"recipe": raw_text})

print(json.dumps(recipe_dict, indent=2))

{
  "title": "Garlic Butter Shrimp",
  "ingredient_names": [
    "shrimp",
    "butter",
    "garlic cloves",
    "lemon",
    "salt",
    "pepper",
    "parsley"
  ],
  "measurements": [
    "200g",
    "2 tbsp",
    "3",
    "half",
    "",
    "",
    ""
  ],
  "directions": [
    "Heat butter in a skillet over medium heat.",
    "Add garlic, cook 30 seconds until fragrant.",
    "Toss in shrimp, season with salt & pepper.",
    "Cook 2\u20133 minutes per side until pink.",
    "Squeeze lemon juice on top and sprinkle parsley.",
    "Serve with rice, pasta, or crusty bread."
  ],
  "total_time": 10,
  "serves": 2
}


## Check Weight Ingredient Tool

In [9]:
# Weight Calculator Tool

from recipe_wrangler.tools.ingredient_weight_tool import ingredient_weight_tool_usda

ingredient_names = recipe_dict['ingredient_names']
measurements = recipe_dict['measurements']

weights = ingredient_weight_tool_usda.invoke({
    "ingredient_names": ingredient_names,
    "measurements": measurements,
    "return_details": True
})
print(weights)


{'weights': [200.0, 28.4, 0.0, 0.0, 0.0, 0.0, 0.0], 'details': [{'name': 'shrimp', 'measurement_raw': '200g', 'parsed_quantity': '200', 'parsed_unit': 'g', 'quantity_inferred': False, 'unit_inferred': False, 'usda_id': '02044', 'portion_match': None, 'match_type': 'direct_mass', 'weight_grams': 200.0, 'error': None}, {'name': 'butter', 'measurement_raw': '2 tbsp', 'parsed_quantity': '2', 'parsed_unit': 'tbsp', 'quantity_inferred': False, 'unit_inferred': False, 'usda_id': '01001', 'portion_match': {'portion_seq': 2, 'amount': 1.0, 'portion_desc': 'tbsp', 'grams': 14.2, 'grams_per_unit': 14.2}, 'match_type': 'direct', 'weight_grams': 28.4, 'error': None}, {'name': 'garlic cloves', 'measurement_raw': '3', 'parsed_quantity': '3', 'parsed_unit': 'clove', 'quantity_inferred': False, 'unit_inferred': True, 'usda_id': '02044', 'portion_match': None, 'match_type': None, 'weight_grams': None, 'error': 'no_weight_match'}, {'name': 'lemon', 'measurement_raw': 'half', 'parsed_quantity': '0.5', 'pa

## Check Embeddings Tool

In [10]:
# Embeddings tool

from recipe_wrangler.tools.ingredient_embeddings_tool import ensure_ingredients_in_collection
payload = ensure_ingredients_in_collection.invoke({
    "ingredient_names": ["garlic", "olive oil", "pasta", "low-fat gouda", "canned cherries"],
    "state": {"persist_path": "chroma_db", "collection_name": "ingredients", "debug": True}
})

print(payload)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


[ensure_ingredients_in_collection] persist_path=chroma_db collection=ingredients
[input] 5 names
[existing] found=5 missing=0
[done] found=5 created=0 failed=0 total=4076
{'persist_path': 'chroma_db', 'collection': 'ingredients', 'found': ['canned cherries', 'garlic', 'low-fat gouda', 'olive oil', 'pasta'], 'created': [], 'failed': [], 'total_in_collection_after': 4076}


In [ ]:
# Check the new added ingredient in the ingredient collection

from recipe_wrangler.utils.query_chromadb import query_sustainability_db, query_nutritional_db_irish, query_ingredients_db

query_ingredients_db("canned cherries")

## Check Sustainability Tool

In [11]:
from recipe_wrangler.tools.sustainability_calculator import sustainability_tool_chroma

In [13]:
# --- Run a real test call via the tool interface (.invoke) ---

result = sustainability_tool_chroma.invoke({
    "title": "Pasta al Pomodoro (Test)",
    "ingredient_names": [
        "tomatoe",      # typo on purpose (should match tomato)
        "olive oil",
        "onion",
        "garlic",
        "spaghetti",
        "parmesan",
        "basil",
    ],
    "measurements": [
        "400.0g",      # typo on purpose (should match tomato)
        "30.0g",
        "80.0g",
        "10.0g",
        "320.0g",
        "40.0g",
        "5.0g",
    ],
    "weights": [
        400.0,      # typo on purpose (should match tomato)
        30.0,
        80.0,
        10.0,
        320.0,
        40.0,
        5.0,
    ],
    "serving_size_g": 250.0,
    "serves": 4,
    "min_similarity": 0.5,      # your requested similarity threshold
})


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 1536.74it/s, Materializing param=norm.weight]                              
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event Clien

In [14]:
result

{'title': 'Pasta al Pomodoro (Test)',
 'details': [{'ingredient': 'tomatoe',
   'matched_sustainability_ingredient': 'tomatoe',
   'weight_g': 400.0,
   'cf_val': 0.48,
   'distance': None,
   'contribution': 0.192,
   'source_sustainability': 'Sustainable FooDB'},
  {'ingredient': 'olive oil',
   'matched_sustainability_ingredient': 'olive oil',
   'weight_g': 30.0,
   'cf_val': 3.84,
   'distance': None,
   'contribution': 0.1152,
   'source_sustainability': 'Sustainable FooDB'},
  {'ingredient': 'onion',
   'matched_sustainability_ingredient': 'onion',
   'weight_g': 80.0,
   'cf_val': 0.24,
   'distance': None,
   'contribution': 0.0192,
   'source_sustainability': 'Sustainable FooDB'},
  {'ingredient': 'garlic',
   'matched_sustainability_ingredient': 'garlic',
   'weight_g': 10.0,
   'cf_val': 0.67,
   'distance': None,
   'contribution': 0.0067,
   'source_sustainability': 'Sustainable FooDB'},
  {'ingredient': 'spaghetti',
   'matched_sustainability_ingredient': 'spaghetti',
  

## Check Nutritional Tool

In [15]:
from recipe_wrangler.tools.nutritional_calculator import nutritional_tool_chroma

In [16]:
# --- Run a real test call via the tool interface (.invoke) ---

result = nutritional_tool_chroma.invoke({
    "title": "Pasta al Pomodoro (Test)",
    "ingredient_names": [
        "tomatoe",      # typo on purpose (should match tomato)
        "olive oil",
        "onion",
        "garlic",
        "spaghetti",
        "parmesan",
        "basil",
    ],
    "measurements": [
        "400.0g",      # typo on purpose (should match tomato)
        "30.0g",
        "80.0g",
        "10.0g",
        "320.0g",
        "40.0g",
        "5.0g",
    ],
    "weights": [
        400.0,      # typo on purpose (should match tomato)
        30.0,
        80.0,
        10.0,
        320.0,
        40.0,
        5.0,
    ],
    "serving_size_g": 250.0,
    "serves": 4,
    "min_similarity": 0.5,      # your requested similarity threshold
})


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes

In [17]:
result

{'title': 'Pasta al Pomodoro (Test)',
 'details': [{'ingredient': 'tomatoe',
   'source': 'Irish Composition Table',
   'source_nutrition': 'Irish Composition Table',
   'matched_nutritional_ingredient': 'Tomatoes, standard, raw',
   'weight_g': 400.0,
   'protein_per_100g': 0.5,
   'carbs_per_100g': 3.0,
   'fat_per_100g': 0.1,
   'sugars_per_100g': 0.0,
   'saturated_fat_per_100g': 0.0,
   'sodium_per_100g_mg': 2.0,
   'protein_g': 2.0,
   'carbs_g': 12.0,
   'fat_g': 0.4,
   'sugar_g': 0.0,
   'saturated_fat_g': 0.0,
   'sodium_mg': 8.0,
   'energy_kcal_per_100g': 0.0,
   'energy_kcal': 0.0,
   'distance': 0.1736053228378296},
  {'ingredient': 'olive oil',
   'source': 'Irish Composition Table',
   'source_nutrition': 'Irish Composition Table',
   'matched_nutritional_ingredient': 'Oil, coconut',
   'weight_g': 30.0,
   'protein_per_100g': 0.0,
   'carbs_per_100g': 0.0,
   'fat_per_100g': 99.9,
   'sugars_per_100g': 0.0,
   'saturated_fat_per_100g': 0.0,
   'sodium_per_100g_mg': 0.0

## Check Profiling Tool

In [2]:
from recipe_wrangler.tools.recipe_profiling_tool import Recipe_Profiling_Tool

2026-01-29 14:54:36.750936757 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
/home/karvanitis/RecipeWrangler-Backend/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
payload = {
    "title": "Pasta al Pomodoro (Test)",
    "ingredient_names": [
        "tomatoe",
        "olive oil",
        "onion",
        "garlic",
        "spaghetti",
        "parmesan",
        "basil",
    ],
    "measurements": ["400.0g", "30.0g", "80.0g", "10.0g", "320.0g", "40.0g", "5.0g"],
    "weights": [400.0, 30.0, 80.0, 10.0, 320.0, 40.0, 5.0],
    "serving_size_g": 250.0,
    "serves": 4,
    "min_similarity": 0.5,
}

profile = Recipe_Profiling_Tool(payload)
from pprint import pprint
pprint(profile)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes

{'ingredients': [{'carbs_g': 12.0,
                  'carbs_per_100g': 3.0,
                  'cf_val': 0.48,
                  'contribution': 0.192,
                  'distance': 0.1736053228378296,
                  'energy_kcal': 0.0,
                  'energy_kcal_per_100g': 0.0,
                  'fat_g': 0.4,
                  'fat_per_100g': 0.1,
                  'ingredient': 'tomatoe',
                  'matched_nutritional_ingredient': 'Tomatoes, standard, raw',
                  'matched_sustainability_ingredient': 'tomatoe',
                  'protein_g': 2.0,
                  'protein_per_100g': 0.5,
                  'saturated_fat_g': 0.0,
                  'saturated_fat_per_100g': 0.0,
                  'sodium_mg': 8.0,
                  'sodium_per_100g_mg': 2.0,
                  'source': 'Irish Composition Table',
                  'source_nutrition': 'Irish Composition Table',
                  'source_sustainability': 'Sustainable FooDB',
                  's

## Check Profiling Chain

In [4]:
from recipe_wrangler.tools.recipe_profiling_chain import Recipe_Profiling_Chain

In [5]:
sample_recipe = """
Pasta al Pomodoro
Serves 4

Ingredients:
- 400 g tomatoes (ripe)
- 30 g olive oil
- 1 small carrot, finely chopped
- 80 g onion, finely chopped
- 10 g garlic, minced
- 320 g spaghetti
- 40 g parmesan, grated
- 5 g basil leaves
- Salt to taste

Directions:
1) Sweat onion in olive oil until translucent. Add garlic briefly.
2) Add chopped tomatoes, simmer 15–20 min. Season with salt.
3) Cook spaghetti al dente; toss with sauce.
4) Serve with basil and parmesan.
"""

res = Recipe_Profiling_Chain.invoke({"recipe_text": sample_recipe, "debug": True})
print(res.keys())              
print(res.get("recipe_profile_totals"))

[Recipe_Parser_Node] Updated State Keys: ['raw_recipe', 'title', 'ingredient_names', 'measurements', 'weights', 'ingredients', 'debug', 'directions', 'total_time', 'tags', 'allergens', 'sustainability_per_kg', 'total_protein_g', 'total_fat_g', 'total_carbohydrate_g', 'total_energy_kcal', 'profiling_totals', 'full_profile', 'serves', 'serving_size_g', 'min_similarity', 'similar_recipes', 'agent_decision', 'query', 'cypher', 'tag_list']


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given



[Ingredient_Weight_Node] Used USDA weights to estimate grams.
[Ingredient_Weight_Node] Updated State Keys: ['raw_recipe', 'title', 'ingredient_names', 'measurements', 'weights', 'ingredients', 'debug', 'directions', 'total_time', 'tags', 'allergens', 'sustainability_per_kg', 'total_protein_g', 'total_fat_g', 'total_carbohydrate_g', 'total_energy_kcal', 'profiling_totals', 'full_profile', 'serves', 'serving_size_g', 'min_similarity', 'similar_recipes', 'agent_decision', 'query', 'cypher', 'tag_list']


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes

[Tagger_Node] Tags calculated for recipe: Pasta al Pomodoro
[Tagger_Node] Updated State Keys: ['raw_recipe', 'title', 'ingredient_names', 'measurements', 'weights', 'ingredients', 'debug', 'directions', 'total_time', 'tags', 'allergens', 'sustainability_per_kg', 'total_protein_g', 'total_fat_g', 'total_carbohydrate_g', 'total_energy_kcal', 'profiling_totals', 'full_profile', 'serves', 'serving_size_g', 'min_similarity', 'similar_recipes', 'agent_decision', 'query', 'cypher', 'tag_list']
[Tagger_Node] Allergens detected: ['Milk', 'Wheat']
dict_keys(['title', 'ingredient_names', 'measurements', 'weights', 'ingredients', 'debug', 'directions', 'total_time', 'tags', 'allergens', 'sustainability_per_kg', 'total_protein_g', 'total_fat_g', 'total_carbohydrate_g', 'total_energy_kcal', 'profiling_totals', 'full_profile', 'serves', 'serving_size_g', 'min_similarity', 'similar_recipes', 'agent_decision', 'query', 'cypher', 'tag_list'])
None


In [6]:
res

{'title': 'Pasta al Pomodoro',
 'ingredient_names': ['tomatoes',
  'olive oil',
  'carrot',
  'onion',
  'garlic',
  'spaghetti',
  'parmesan',
  'basil leaves',
  'Salt'],
 'measurements': ['400 g',
  '30 g',
  '1 small',
  '80 g',
  '10 g',
  '320 g',
  '40 g',
  '5 g',
  'to taste'],
 'weights': [400.0, 30.0, 0.0, 80.0, 10.0, 320.0, 0.0, 5.0, 0.0],
 'ingredients': [{'name': 'tomatoes',
   'measurement': '400 g',
   'weight_g': 400.0,
   'source': 'Irish Composition Table',
   'matched_nutritional_ingredient': 'Tomatoes, standard, raw',
   'protein_per_100g': 0.5,
   'carbs_per_100g': 3.0,
   'fat_per_100g': 0.1,
   'protein_g': 2.0,
   'carbs_g': 12.0,
   'fat_g': 0.4,
   'distance': 0.13727879524230957,
   'sustainability_ingredient': None,
   'matched_sustainability_ingredient': 'tomato',
   'sustainability_weight_g': 400.0,
   'cf_val': 0.48,
   'sustainability_distance': None,
   'contribution': 0.192},
  {'name': 'olive oil',
   'measurement': '30 g',
   'weight_g': 30.0,
   's

## LangChain T2C tool

In [ ]:
from recipe_wrangler.tools.text2cypher import RecipeSearchApp
import os

app = RecipeSearchApp(
    neo4j_uri=os.environ["NEO4J_URI"]
)

result = app.invoke("Give me a low fat recipe with chicken under 120 minutes")
print(result['results'])
print(result['cypher_statement'])

## Fetch Recipe Info

In [ ]:
from recipe_wrangler.tools.fetch_recipe_info import fetch_recipe_info

info = fetch_recipe_info("Cajun Chicken Salad")

print(info)

# Test LangChain T2C Components

In [ ]:
import os

from recipe_wrangler.tools.text2cypher import RecipeSearchApp

app = RecipeSearchApp(
    neo4j_uri=os.environ["NEO4J_URI"]
)


In [ ]:
app.invoke("chicken and rice under 30 min")